# Cadrage : entraînement en parties

Un seul entraînement, en **4 parties qui se suivent**. Chaque partie est sauvegardée sur **ton Google Drive**.

**Avant tout :** menu **Exécution → Modifier le type d'exécution → T4 GPU → Enregistrer**.

1. Lance **Préparer** (à refaire à chaque nouvelle session Colab).
2. Lance **Partie 0**. Si elle échoue, ne va pas plus loin.
3. Lance les parties **1, 2, 3, 4** dans l'ordre, le même jour ou non.

Si Colab coupe : relance **Préparer**, puis la même partie. N'utilise pas « Tout exécuter ».

Données : uniquement des documents publics (CC BY et décision 2011/833/UE). **Jamais de document client.**
Projet étudiant, non affilié à EY.

In [ ]:
#@title Préparer (à chaque nouvelle session) {display-mode: "form"}
import hashlib, os, shutil, signal, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path

CODE = "8e8ecd47d21d25121cbe7f20bc47f449f36521c6"
LLAMA = "e300919cc704ddde3a8a8346a8714f57eeeee71b"
DONNEES = "24b2bb02b7f0fcf6595aff87126d764ed9a6d7f431bb33104ea7eb4e83cc3150"
OLLAMA = "0.40.16"  # la même version que sur le PC

def sh(commande):
    r = subprocess.run(["bash", "-c", "set -o pipefail; " + commande], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"Échec : {commande}\n{(r.stderr or r.stdout)[-1500:]}")
    return r.stdout

if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode:
    raise RuntimeError("Pas de GPU. Menu Exécution, Modifier le type d'exécution, T4 GPU, puis relance.")
print("GPU :", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").strip())

from google.colab import drive, files
drive.mount("/content/drive")
STOCK = Path("/content/drive/MyDrive/cadrage-entrainement")
STOCK.mkdir(parents=True, exist_ok=True)

ARCHIVE = STOCK / "cadrage-donnees-publiques.zip"
if not ARCHIVE.exists():
    print("Choisis cadrage-donnees-publiques.zip (dossier training/data du projet sur ton PC)")
    recu = files.upload()
    contenu = next(iter(recu.values()), b"")
    if hashlib.sha256(contenu).hexdigest() != DONNEES:
        raise RuntimeError("Ce n'est pas le bon fichier de données. Relance la cellule.")
    ARCHIVE.write_bytes(contenu)
if hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() != DONNEES:
    raise RuntimeError("Le fichier de données sur Drive a changé : supprime-le et relance.")

print("1/5 code")
os.chdir("/content")
REPO = Path("/content/cadrage")
sh(f"rm -rf {REPO} && git init -q {REPO} && git -C {REPO} fetch -q --depth 1 "
   f"https://github.com/YoussefBY22/PRV-PR8-PFE-IA-Cadrage.git {CODE} && git -C {REPO} checkout -q FETCH_HEAD")
zipfile.ZipFile(ARCHIVE).extractall(REPO / "training/data/real")

print("2/5 bibliothèques")
sh(f"{sys.executable} -m pip install -q transformers==4.57.6 peft==0.17.1 accelerate==1.10.1 "
   "bitsandbytes==0.50.2 huggingface_hub==0.36.2 sentencepiece pyyaml")
sh("rm -rf /content/llama.cpp && git init -q /content/llama.cpp && git -C /content/llama.cpp fetch -q --depth 1 "
   f"https://github.com/ggml-org/llama.cpp.git {LLAMA} && git -C /content/llama.cpp checkout -q FETCH_HEAD")

print("3/5 modèle de base")
BASE = sh(f"""{sys.executable} -c 'from huggingface_hub import snapshot_download as d; """
          """print(d("Qwen/Qwen3-1.7B", revision="70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"))'""").strip().splitlines()[-1]

print("4/5 Ollama")
if not shutil.which("ollama"):
    sh("apt-get install -y -qq zstd")
    sh(f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA} sh")
def ollama_pret():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        return True
    except OSError:
        return False
if not ollama_pret():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(60):
        if ollama_pret():
            break
        time.sleep(1)
print("5/5 gabarit du modèle")
sh("ollama pull qwen3:1.7b")

def cadrage(*action):
    env = dict(os.environ, LLAMA_CPP_DIR="/content/llama.cpp", PYTHONUNBUFFERED="1",
               HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    cmd = [sys.executable, "-m", "training.parts", *map(str, action),
           "--store", str(STOCK / "etat-v1"), "--base-dir", BASE]
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors="replace", start_new_session=True)
    try:
        for ligne in p.stdout:
            print(ligne, end="")
    except KeyboardInterrupt:
        print("Arrêt demandé : sauvegarde en cours, attends le message de fin.")
        p.send_signal(signal.SIGINT)
        for ligne in p.stdout:
            print(ligne, end="")
    code = p.wait()
    os.sync()
    if code:
        raise RuntimeError(f"Non terminé (code {code}). Copie ce qui est affiché au-dessus à Claude.")

cadrage("etat")
print("PRÊT.")

In [ ]:
#@title Partie 0 : preuve (obligatoire) {display-mode: "form"}
cadrage("preuve")

In [ ]:
#@title Partie 1 : pas 1 à 50 {display-mode: "form"}
cadrage("partie", 1)

In [ ]:
#@title Partie 2 : pas 51 à 100 {display-mode: "form"}
cadrage("partie", 2)

In [ ]:
#@title Partie 3 : pas 101 à 150 {display-mode: "form"}
cadrage("partie", 3)

In [ ]:
#@title Partie 4 : pas 151 à 200 {display-mode: "form"}
cadrage("partie", 4)

In [ ]:
#@title Où j'en suis {display-mode: "form"}
cadrage("etat")

In [ ]:
#@title Exporter le modèle (après la partie 2 ou 4, environ 4 Go sur Drive) {display-mode: "form"}
cadrage("exporter")